In [1]:
import os

os.environ["TRANSFORMERS_CACHE"] = "D:/huggingface_cache"
os.environ["HF_HOME"] = "D:/huggingface_cache"


In [2]:
!pip install transformers datasets torch scikit-learn --upgrade

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load datasets
def load_data(base_path):
    train = pd.read_csv(f"{base_path}/train.csv")
    dev = pd.read_csv(f"{base_path}/dev.csv") 
    test = pd.read_csv(f"{base_path}/test.csv")
    return train, dev, test

base_path = r"C:\Users\Harshita\OneDrive\Desktop\Med-MMHL\fakenews_article"
train_df, dev_df, test_df = load_data(base_path)

# Verify data
print(f"Train: {len(train_df)}, Dev: {len(dev_df)}, Test: {len(test_df)}")
print("Sample:\n", train_df[["content", "det_fake_label"]].head())

Train: 8309, Dev: 1189, Test: 2375
Sample:
                                              content  det_fake_label
0  "Five photos have been shared thousands of tim...               1
1  The development of effective anti-androgen the...               0
2  "\nThis presentation is particularly unfortuna...               0
3  Oct. 7, 2022 – Moms who consume ultra-processe...               0
4  Why do some people with cold sores around thei...               0


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "dmis-lab/biobert-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    problem_type="single_label_classification"
)

C:\Users\Harshita\AppData\Roaming\Python\Python313\site-packages\transformers\utils\hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
from datasets import Dataset

def prepare_datasets(df):
    return Dataset.from_pandas(df[["content", "det_fake_label"]])

train_ds = prepare_datasets(train_df)
dev_ds = prepare_datasets(dev_df)
test_ds = prepare_datasets(test_df)

def tokenize(batch):
    return tokenizer(
        batch["content"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

tokenized_train = train_ds.map(tokenize, batched=True)
tokenized_dev = dev_ds.map(tokenize, batched=True)
tokenized_test = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/8309 [00:00<?, ? examples/s]

Map:   0%|          | 0/1189 [00:00<?, ? examples/s]

Map:   0%|          | 0/2375 [00:00<?, ? examples/s]

In [6]:
!pip install transformers datasets torch scikit-learn --upgrade

Defaulting to user installation because normal site-packages is not writeable


In [7]:
!pip install --upgrade transformers


Defaulting to user installation because normal site-packages is not writeable


In [8]:
print(f"Training samples: {len(tokenized_train)}")
print(f"Validation samples: {len(tokenized_dev)}")
print("Sample check:", tokenized_train[0]['input_ids'][:10])  # Show first 10 tokens

Training samples: 8309
Validation samples: 1189
Sample check: [101, 107, 4222, 7630, 1138, 1151, 3416, 4674, 1104, 1551]


In [9]:
import transformers
print(transformers.__version__)


4.51.3


In [10]:
!pip uninstall -y transformers
!pip install transformers==4.51.3


Found existing installation: transformers 4.51.3
Uninstalling transformers-4.51.3:
  Successfully uninstalled transformers-4.51.3
Defaulting to user installation because normal site-packages is not writeable
  Using cached transformers-4.51.3-py3-none-any.whl.metadata (38 kB)
Using cached transformers-4.51.3-py3-none-any.whl (10.4 MB)


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [11]:
import transformers
print(transformers.__version__)


4.51.3


In [15]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score

# Define the compute_metrics function
training_args = TrainingArguments(
    output_dir="D:/bert_checkpoints",  # Directory to save checkpoints
    per_device_train_batch_size=16,  # Batch size for training
    per_device_eval_batch_size=16,  # Batch size for evaluation
    num_train_epochs=3,  # Number of epochs
    logging_dir="D:/bert_logs",  # Directory for logs
    logging_steps=10,  # Log every 10 steps
    load_best_model_at_end=True,  # Load best model after training
    metric_for_best_model="accuracy",  # Metric to decide best model
    greater_is_better=True,  # Specify that a higher accuracy is better
    save_strategy="epoch",  # Save after every epoch
    evaluation_strategy="epoch"  # Evaluate after every epoch
)

# Ensure your model is defined (this is just an example)
from transformers import BertForSequenceClassification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)  # num_labels depends on your task

# Ensure tokenized datasets are defined and have labels
tokenized_train = ...  # Tokenized training dataset with labels
tokenized_dev = ...  # Tokenized validation dataset with labels

# Ensure your tokenizer is defined
tokenizer = ...  # Tokenizer associated with your model

# Create the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    compute_metrics=compute_metrics,  # Provide the metrics function
    tokenizer=tokenizer  # Use the tokenizer for encoding
)

# Train the model
trainer.train()


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'